In [1]:
import requests
import pandas as pd
import time
#OpenAlex API
base_url = "https://api.openalex.org/works"
#查询参数
params = {
    # 检索标题和摘要中的 ADHD
    "search.title_and_abstract": "ADHD",
    # 筛选条件
    "filter": (
        "type:article,"
        "publication_year:2023-2025,"
        "primary_topic.field.id:27"
    ),
    # 每页最大 200
    "per-page": 200,
    # 游标翻页
    "cursor": "*"
}
#保存结果
all_papers = []
print("开始下载 ADHD 文献数据...")
#开始爬取
while True:
    try:
        response = requests.get(
            base_url,
            params=params,
            timeout=60
        )
        # 请求失败
        if response.status_code != 200:
            print("请求失败：", response.status_code)
            break
        result = response.json()
        page_items = result.get("results", [])
        # 没有数据则结束
        if not page_items:
            break
        #遍历每篇论文
        for item in page_items:
            # ---------------------------------
            # 摘要
            # ---------------------------------
            abstract = ""
            inverted_index = item.get(
                "abstract_inverted_index"
            )
            if inverted_index:
                word_list = []
                for word, positions in inverted_index.items():
                    for pos in positions:
                        word_list.append(
                            (pos, word)
                        )
                word_list.sort()
                abstract = " ".join(
                    [
                        word
                        for pos, word
                        in word_list
                    ]
                )
            # ---------------------------------
            # 作者 / 机构 / 国家
            # ---------------------------------
            authors = []
            institutions = []
            countries = []
            authorships = item.get(
                "authorships"
            )
            if authorships:
                for auth in authorships:
                    # 作者
                    author_info = auth.get(
                        "author"
                    )
                    if (
                        author_info
                        and author_info.get(
                            "display_name"
                        )
                    ):
                        authors.append(
                            str(
                                author_info.get(
                                    "display_name"
                                )
                            )
                        )
                    # 机构
                    insts = auth.get(
                        "institutions"
                    )
                    if insts:
                        for inst in insts:
                            inst_name = inst.get(
                                "display_name"
                            )
                            if inst_name:
                                institutions.append(
                                    str(inst_name)
                                )
                            country = inst.get(
                                "country_code"
                            )
                            if country:
                                countries.append(
                                    str(country)
                                )
            # 去重
            authors = list(set(authors))
            institutions = list(
                set(institutions)
            )
            countries = list(set(countries))
            # ---------------------------------
            # Concepts
            # ---------------------------------
            concepts = []
            concept_data = item.get(
                "concepts"
            )
            if concept_data:
                concepts = [
                    str(
                        c.get(
                            "display_name"
                        )
                    )
                    for c in concept_data
                    if c.get("display_name")
                ]
            # ---------------------------------
            # Topics
            # ---------------------------------
            topics = []
            topic_data = item.get(
                "topics"
            )
            if topic_data:
                topics = [
                    str(
                        t.get(
                            "display_name"
                        )
                    )
                    for t in topic_data
                    if t.get("display_name")
                ]
            # ---------------------------------
            # Journal
            # ---------------------------------
            journal = ""
            primary_location = item.get(
                "primary_location"
            )
            if (
                primary_location
                and primary_location.get(
                    "source"
                )
            ):
                source = primary_location.get(
                    "source"
                )
                if source:
                    journal_name = source.get(
                        "display_name"
                    )
                    if journal_name:
                        journal = str(
                            journal_name
                        )
            # ---------------------------------
            # Open Access
            # ---------------------------------
            is_oa = False
            open_access = item.get(
                "open_access"
            )
            if open_access:
                is_oa = open_access.get(
                    "is_oa",
                    False
                )
            # ---------------------------------
            # 引用百分位
            # ---------------------------------
            citation_percentile = None
            percentile_data = item.get(
                "cited_by_percentile_year"
            )
            if percentile_data:
                citation_percentile = (
                    percentile_data.get(
                        "max"
                    )
                )
            # ---------------------------------
            # 最近一年引用
            # ---------------------------------
            recent_citations = 0
            counts_by_year = item.get(
                "counts_by_year"
            )
            if counts_by_year:
                recent_citations = (
                    counts_by_year[0].get(
                        "cited_by_count",
                        0
                    )
                )
            # ---------------------------------
            # 参考文献数量
            # ---------------------------------
            referenced_works_count = len(
                item.get(
                    "referenced_works",
                    []
                )
            )
            # ---------------------------------
            # 文本长度特征
            # ---------------------------------
            title = item.get("title", "")
            title_length = len(title)
            abstract_length = len(abstract)
            # ---------------------------------
            # 特征统计
            # ---------------------------------
            concepts_count = len(concepts)
            topics_count = len(topics)
            authors_count = len(authors)
            institutions_count = len(
                institutions
            )
            countries_count = len(
                countries
            )
            # ---------------------------------
            # 国际合作
            # ---------------------------------
            international_collab = (
                1
                if countries_count > 1
                else 0
            )
            # ---------------------------------
            # 多机构合作
            # ---------------------------------
            multi_institution = (
                1
                if institutions_count > 1
                else 0
            )
            # ---------------------------------
            # 高被引标签
            # 前10%作为高被引
            # ---------------------------------
            is_highly_cited = (
                1
                if (
                    citation_percentile
                    and citation_percentile >= 90
                )
                else 0
            )
            # ---------------------------------
            # 保存数据
            # ---------------------------------
            all_papers.append({
                # 基础信息
                "paper_id":
                    item.get("id"),
                "title":
                    title,
                "abstract":
                    abstract,
                "publication_year":
                    item.get(
                        "publication_year"
                    ),
                "doi":
                    item.get("doi"),
                # 期刊
                "journal":
                    journal,
                # 被引
                "cited_by_count":
                    item.get(
                        "cited_by_count"
                    ),
                "citation_percentile":
                    citation_percentile,
                "recent_citations":
                    recent_citations,
                # 高被引标签
                "is_highly_cited":
                    is_highly_cited,
                # OA
                "is_oa":
                    is_oa,
                # 作者
                "authors":
                    "; ".join(authors),
                "authors_count":
                    authors_count,
                # 机构
                "institutions":
                    "; ".join(
                        institutions
                    ),
                "institutions_count":
                    institutions_count,
                # 国家
                "countries":
                    "; ".join(countries),
                "countries_count":
                    countries_count,
                # 合作特征
                "international_collab":
                    international_collab,
                "multi_institution":
                    multi_institution,
                # 概念
                "concepts":
                    "; ".join(concepts),
                "concepts_count":
                    concepts_count,
                # 主题
                "topics":
                    "; ".join(topics),
                "topics_count":
                    topics_count,
                # 文本长度
                "title_length":
                    title_length,
                "abstract_length":
                    abstract_length,
                # 参考文献数量
                "referenced_works_count":
                    referenced_works_count
            })
        # 输出进度
        print(
            f"已下载：{len(all_papers)} 条"
        )
        # 下一页
        next_cursor = result.get(
            "meta",
            {}
        ).get("next_cursor")
        if not next_cursor:
            break
        params["cursor"] = next_cursor
        # 防止请求过快
        time.sleep(1)
    except Exception as e:
        print("发生错误：", e)
        break
#转 DataFrame
df = pd.DataFrame(all_papers)
#删除重复 DOI
df = df.drop_duplicates(
    subset="doi"
)
#删除标题为空
df = df.dropna(
    subset=["title"]
)
#保存CSV
df.to_csv(
    "ADHD_OpenAlex数据集.csv",
    index=False,
    encoding="utf-8-sig"
)
#输出结果
print("\n数据下载完成！")
print(f"最终文献数：{len(df)}")
print(
    "文件已保存：ADHD_OpenAlex数据集.csv"
)

C:\Users\21116\AppData\Roaming\Python\Python313\site-packages\pandas\core\computation\expressions.py:22: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.10.1' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED


开始下载 ADHD 文献数据...
已下载：200 条
已下载：400 条
已下载：600 条
已下载：800 条
已下载：1000 条
已下载：1200 条
已下载：1400 条
已下载：1600 条
已下载：1800 条
已下载：2000 条
已下载：2200 条
已下载：2400 条
已下载：2600 条
已下载：2800 条
已下载：3000 条
已下载：3200 条
已下载：3400 条
已下载：3600 条
已下载：3800 条
已下载：4000 条
已下载：4200 条
已下载：4400 条
已下载：4600 条
已下载：4800 条
已下载：5000 条
已下载：5200 条
已下载：5400 条
已下载：5600 条
已下载：5800 条
已下载：6000 条
已下载：6200 条
已下载：6400 条
已下载：6600 条
已下载：6800 条
已下载：7000 条
已下载：7200 条
已下载：7400 条
已下载：7600 条
已下载：7800 条
已下载：8000 条
已下载：8200 条
已下载：8400 条
已下载：8600 条
已下载：8668 条

数据下载完成！
最终文献数：8338
文件已保存：ADHD_OpenAlex数据集.csv
